Metrics (1)

- mean_surprisal — mean token surprisal under GPT-2, computed per sentence and then averaged over sentences

In [8]:
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
import torch
import math
import spacy

# GPT-2 small - surprisal of every token given all tokens before it in the paragraph
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.to("mps")
model.eval() # Set the model to evaluation mode i.e. inference mode

# en_core_web_trf is used only to split paragraph into sentences - same sentence boundaries as mean-mdl.ipynb
# sentences come from the parser, so tagger, attribute_ruler, ner and lemmatizer are disabled
spacy.prefer_gpu() # uses the Mac GPU (MPS) if available
nlp = spacy.load("en_core_web_trf", disable=["tagger", "attribute_ruler", "ner", "lemmatizer"])
print(f"Language model: {nlp.meta['lang']}_{nlp.meta['name']}")
print(f"Version: {nlp.meta['version']}")
print(f"Pipe names: {nlp.pipe_names}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7305.00it/s]


Language model: en_core_web_trf
Version: 3.8.0
Pipe names: ['transformer', 'parser']


In [2]:
def get_sentences(doc):
    #paragraph is split into sentences using the parser (same as mean-mdl.ipynb)
    return list(doc.sents)

In [3]:
def token_surprisals(text):
    #surprisal of each token = -log2 P(token | all previous tokens of the paragraph)
    #whole paragraph goes to GPT-2 in one pass, so tokens of later sentences still see earlier sentences as context
    encoding = tokenizer(text, truncation=True, max_length=1024, return_offsets_mapping=True)
    token_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]
    if LEVEL == "sentence": #LEVEL is set in the paths cell below
       #a sentence is short, so losing its first token matters - put <|endoftext|> in front
       #so the first token also has a context and gets a surprisal; the start token itself is never scored
       token_ids = [tokenizer.bos_token_id] + token_ids[:1023]
       offsets = [(0, 0)] + offsets[:1023]
    if len(token_ids) < 2: #need at least one previous token to predict from
       return []
    input_ids = torch.tensor([token_ids], device="mps")
    with torch.no_grad():
        logits = model(input_ids).logits[0]

    #logits at position i predict token i+1, so first token gets no surprisal (no context before it)
    next_token_probs = torch.softmax(logits[:-1], dim=-1)
    next_token_ids = input_ids[0, 1:].unsqueeze(1)
    surprisals = (-torch.log2(next_token_probs.gather(1, next_token_ids).squeeze(1))).tolist()

    scored_tokens = [] #list of (char position where token starts, surprisal, token text) for token 1 .. n-1
    for index, surprisal in enumerate(surprisals, start=1):
        start, end = offsets[index]
        token_text = text[start:end]
        #GPT-2 token carries the space before the word (" His"), so skip the space to find where the word really starts
        start += len(token_text) - len(token_text.lstrip())
        scored_tokens.append((start, surprisal, tokenizer.decode([token_ids[index]])))
    return scored_tokens

In [4]:
def sentence_surprisal(scored_tokens, start_char, end_char):
    #a GPT-2 token belongs to this sentence if its word starts inside the sentence characters
    sentence_values = [surprisal for char_start, surprisal, _ in scored_tokens if start_char <= char_start < end_char]
    if not sentence_values: #no scored token (e.g. first sentence is a single token), so no surprisal for this sentence
       return None
    return sum(sentence_values) / len(sentence_values)

In [5]:
from spacy.tokens import Doc

def calculate_mean_surprisal(text):
    #text can be a raw string or an already parsed doc (from nlp.pipe), so the paragraph is never parsed twice
    doc = text if isinstance(text, Doc) else nlp(text)
    scored_tokens = token_surprisals(doc.text)
    sentences = get_sentences(doc)

    sentence_surprisals = [] #list to store mean surprisal of each sentence
    for index, sent in enumerate(sentences):
        #sentence owns all characters up to start of next sentence, so tokens in the whitespace between sentences are not lost
        start_char = 0 if index == 0 else sent.start_char
        end_char = sentences[index + 1].start_char if index + 1 < len(sentences) else len(doc.text)
        surprisal = sentence_surprisal(scored_tokens, start_char, end_char)
        if surprisal is not None: #skip sentences with no scored tokens
           sentence_surprisals.append(surprisal)

    #paragraph surprisal = unweighted mean of sentence surprisals
    if sentence_surprisals:
        mean_surprisal = sum(sentence_surprisals) / len(sentence_surprisals)
    else:
        mean_surprisal = 0

    return f"{mean_surprisal:.5f}"

In [ ]:
from pathlib import Path
import csv

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/raw-csv to read and data/<LEVEL>/features/ to write
raw_csv_path = Path(f"../../data/{LEVEL}/raw-csv")
# old pooled values (mean over all tokens of paragraph) are archived outside the repo in mtech-thesis/experiments-archive/paragraph/pooled-sm-2026-11-11/
out_csv_path = Path(f"../../data/{LEVEL}/features/mean-surprisal-gpt2-csv")
out_csv_path.mkdir(parents=True, exist_ok=True)

for csv_file in sorted(raw_csv_path.glob("*.csv")):
    out_csv_file = out_csv_path / csv_file.name
    if out_csv_file.exists() and out_csv_file.stat().st_size > 0: # Skip if already exist
       continue
    print(f"Working on file: {csv_file.name}")
    with csv_file.open("r",encoding="utf-8") as cf:
       csv_reader = csv.DictReader(cf)
       column_names = list(csv_reader.fieldnames)
       column_names.insert(-1, "mean_surprisal")
       rows = list(csv_reader)
    texts = [row["text"] for row in rows]
    # write to a temp file and rename at the end, so a half written file is never skipped on restart
    tmp_csv_file = out_csv_file.with_suffix(".csv.tmp")
    with tmp_csv_file.open("w",encoding="utf-8") as of:
       csv_writer = csv.DictWriter(of,fieldnames=column_names)
       csv_writer.writeheader()
       # nlp.pipe parses texts in batches - much faster than calling nlp(text) for every row
       for row, doc in zip(rows, nlp.pipe(texts, batch_size=64)):
          row["mean_surprisal"] = calculate_mean_surprisal(doc)
          csv_writer.writerow(row)
    tmp_csv_file.rename(out_csv_file)
    # break #- During demo we can uncomment and providea demo path

In [6]:
# Test GPT2 surprisal code

text = "For more information about..."
print(calculate_mean_surprisal(text))
text = "Timbaland was one of the most successful producers of the 2000s. His biggest hits include “Promiscuous” by Nelly Furtado, “Work It” by Missy Elliott, and “Cry Me a River” by Justin Timberlake."
print(calculate_mean_surprisal(text))

# per sentence check - paragraph surprisal must be the mean of the sentence surprisals
doc = nlp(text)
scored_tokens = token_surprisals(text)
sentences = get_sentences(doc)
sentence_values = [sentence_surprisal(scored_tokens, 0, sentences[1].start_char),
                   sentence_surprisal(scored_tokens, sentences[1].start_char, len(text))]
print(sentence_values)
assert abs(float(calculate_mean_surprisal(doc)) - sum(sentence_values) / len(sentence_values)) < 1e-4

# every scored token must belong to exactly one sentence (no token lost between sentences)
assert sum(1 for start, _, _ in scored_tokens if start < sentences[1].start_char) + sum(1 for start, _, _ in scored_tokens if start >= sentences[1].start_char) == len(scored_tokens)

# first word of sentence 2 (' His') must be counted in sentence 2, not sentence 1
assert next(start for start, _, tok in scored_tokens if tok == " His") == sentences[1].start_char

# edge cases - nothing to score must give 0, not a division by zero
assert calculate_mean_surprisal("") == "0.00000"
# a single token: at paragraph level it has no context before it, so nothing is scored -> 0;
# at sentence level <|endoftext|> is put in front, so it gets a real surprisal
assert (calculate_mean_surprisal("Hello") == "0.00000") == (LEVEL == "paragraph")
print("all surprisal checks passed")

6.49201
4.20902
[4.381510386509555, 4.036537292318656]
all surprisal checks passed


In [7]:
# DEMO - HOW SENTENCE LEVEL SURPRISAL WORKS
# step 1: whole paragraph goes to GPT-2 once, surprisal of each token = -log2 P(token | all previous tokens)
# step 2: split paragraph into sentences, each token goes to the sentence it starts in, mean = sentence surprisal
# step 3: paragraph surprisal = unweighted mean of sentence surprisals

demo_text = "Timbaland was one of the most successful producers of the 2000s. His biggest hits include “Promiscuous” by Nelly Furtado, “Work It” by Missy Elliott, and “Cry Me a River” by Justin Timberlake."
demo_doc = nlp(demo_text)
demo_scored_tokens = token_surprisals(demo_text)
demo_sentences = get_sentences(demo_doc)
first_token = tokenizer.decode(tokenizer(demo_text)["input_ids"][0])
print(f"first token '{first_token}' has no previous context, so it gets no surprisal")

demo_sentence_surprisals = [] #list to store mean surprisal of each sentence
for index, sent in enumerate(demo_sentences):
    start_char = 0 if index == 0 else sent.start_char
    end_char = demo_sentences[index + 1].start_char if index + 1 < len(demo_sentences) else len(demo_text)

    print(f"\n=== Sentence {index + 1}: {sent.text.strip()}")
    print(f"{'TOKEN':18} {'CHAR':6} {'SURPRISAL':10}")
    values = [] #surprisals of tokens of this sentence
    for char_start, surprisal, token_text in demo_scored_tokens:
        if start_char <= char_start < end_char:
           #GPT-2 token, so it can be part of a word; characters like “ are split into byte pieces and show as �
           print(f"{token_text!r:18} {char_start:<6} {surprisal:<10.4f}")
           values.append(surprisal)

    if values:
        sent_value = sum(values) / len(values)
        demo_sentence_surprisals.append(sent_value)
        print(f"Sentence {index + 1} surprisal = {sum(values):.4f} / {len(values)} = {sent_value:.4f}")
        assert abs(sent_value - sentence_surprisal(demo_scored_tokens, start_char, end_char)) < 1e-9 #same value as the real function
    else:
        print(f"Sentence {index + 1} has no scored tokens, so it is skipped")

#paragraph surprisal = unweighted mean of sentence surprisals
if demo_sentence_surprisals:
    paragraph_surprisal = sum(demo_sentence_surprisals) / len(demo_sentence_surprisals)
else:
    paragraph_surprisal = 0
print(f"\nParagraph surprisal = mean of {[round(v, 4) for v in demo_sentence_surprisals]} = {paragraph_surprisal:.5f}")
print(f"calculate_mean_surprisal gives: {calculate_mean_surprisal(demo_doc)}") #same value as the real function
all_values = [s for _, s, _ in demo_scored_tokens]
print(f"old pooled method (mean over all {len(all_values)} tokens) would give: {sum(all_values) / len(all_values):.5f}")

first token 'Tim' has no previous context, so it gets no surprisal

=== Sentence 1: Timbaland was one of the most successful producers of the 2000s.
TOKEN              CHAR   SURPRISAL 
'bal'              3      13.4638   
'and'              6      2.9154    
' was'             10     5.7366    
' one'             14     5.9642    
' of'              18     0.0708    
' the'             21     0.9379    
' most'            25     3.3743    
' successful'      30     3.7212    
' producers'       41     7.7438    
' of'              51     1.0326    
' the'             54     2.6697    
' 2000'            58     9.6908    
's'                62     1.4648    
'.'                63     2.5553    
Sentence 1 surprisal = 61.3411 / 14 = 4.3815

=== Sentence 2: His biggest hits include “Promiscuous” by Nelly Furtado, “Work It” by Missy Elliott, and “Cry Me a River” by Justin Timberlake.
TOKEN              CHAR   SURPRISAL 
' His'             65     3.5533    
' biggest'         69     9.1114